# EventStudy_RiskModel_DeskAnalytics — Run & Review Notebook

This notebook runs the full **event study → event-conditioned risk → stress** pipeline and then loads the outputs.

## Prereqs
In a terminal (once per environment):
```bash
pip install -r requirements.txt
```


## 1) Configure inputs
- `events_csv`: your event list (`data/events.csv` format: ticker, event_date, event_type)
- `benchmark`: market index ticker (default: `^GSPC`)


In [1]:
events_csv = "../data/events.csv"  # edit if needed
benchmark = "^GSPC"                 # S&P 500
outdir = "../outputs"              # where pipeline writes outputs

print("events_csv:", events_csv)
print("benchmark:", benchmark)
print("outdir:", outdir)


events_csv: ../data/events.csv
benchmark: ^GSPC
outdir: ../outputs


## 2) Run pipeline
This pulls prices from Yahoo Finance (cached under `data/prices_cache/`) and writes CSV outputs to `outputs/`.


In [2]:
import os, sys, subprocess

# If you're inside /notebooks, move to repo root
repo_root = os.path.abspath("..")
os.chdir(repo_root)
print("CWD:", os.getcwd())

cmd = [
    sys.executable, "run_pipeline.py",
    "--events", "data/events.csv",
    "--benchmark", "^GSPC",
    "--outdir", "outputs"
]

res = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", res.stdout)
print("STDERR:\n", res.stderr)

res.check_returncode()
import sys, subprocess, shlex

cmd = f"python ../run_pipeline.py --events {events_csv} --benchmark {benchmark} --outdir {outdir}"
print(cmd)
subprocess.run(shlex.split(cmd), check=True)


CWD: C:\Users\gtmlv
STDOUT:
 
STDERR:
 C:\Users\gtmlv\anaconda3\python.exe: can't open file 'C:\\Users\\gtmlv\\run_pipeline.py': [Errno 2] No such file or directory



CalledProcessError: Command '['C:\\Users\\gtmlv\\anaconda3\\python.exe', 'run_pipeline.py', '--events', 'data/events.csv', '--benchmark', '^GSPC', '--outdir', 'outputs']' returned non-zero exit status 2.

## 3) Load outputs


In [ ]:
import pandas as pd

event_study = pd.read_csv(f"{outdir}/event_study_results.csv")
risk = pd.read_csv(f"{outdir}/risk_metrics.csv")

display(event_study.head())
display(risk.head())


## 4) Quick checks
### 4.1 Event-study CAR summary


In [ ]:
car_cols = [c for c in event_study.columns if c.startswith("CAR_")]
event_study[car_cols].describe().T


### 4.2 Event Risk Multiplier (ERM)
ERM is computed as **Student‑t ES99(event) / Student‑t ES99(normal)**.


In [ ]:
erm = (risk.dropna(subset=["ERM_ES99_student_t"])
       .drop_duplicates(subset=["ticker"])
       [["ticker","ERM_ES99_student_t"]]
       .sort_values("ERM_ES99_student_t", ascending=False))

display(erm)


## 5) Plot ERM


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.bar(erm["ticker"].astype(str), erm["ERM_ES99_student_t"].astype(float))
plt.title("Event Risk Multiplier (ES99, Student‑t)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## 6) Stress results (CAR-tail scenario)
The pipeline also writes `stress_results.csv` when CAR columns exist.


In [ ]:
import os

stress_path = f"{outdir}/stress_results.csv"
if os.path.exists(stress_path):
    stress = pd.read_csv(stress_path, index_col=0)
    display(stress)
else:
    print("No stress_results.csv found.")
